In [6]:
import pandas as pd
import re
from tqdm import tqdm

def categorize_disease(disease_str):
    """Categorize diseases into broader groups."""
    if pd.isna(disease_str):
        return 'Other'
        
    disease_str = disease_str.lower()
    
    # Define disease categories and their related terms
    categories = {
        'Cancer': ['cancer', 'tumor', 'carcinoma', 'sarcoma', 'leukemia', 'lymphoma', 
                  'melanoma', 'neoplasia', 'neoplasm', 'malignant', 'adenoma', 'oncogenic'],
        'Cardiovascular': ['heart', 'cardiac', 'cardiovascular', 'arrhythmia', 'atherosclerosis',
                          'stroke', 'thrombosis', 'aneurysm', 'vascular', 'coronary', 'myocardial'],
        'Neurological': ['brain', 'neural', 'neuro', 'alzheimer', 'parkinson', 'epilepsy', 
                        'seizure', 'dementia', 'cognitive', 'huntington', 'multiple sclerosis'],
        'Metabolic': ['diabetes', 'metabolic', 'obesity', 'thyroid', 'cholesterol', 
                     'lipid', 'insulin', 'glucose'],
        'Autoimmune': ['autoimmune', 'lupus', 'arthritis', 'rheumatoid', 'psoriasis', 
                      'celiac', 'crohn', 'inflammatory bowel', 'multiple sclerosis'],
        'Respiratory': ['lung', 'pulmonary', 'respiratory', 'asthma', 'copd', 'fibrosis',
                       'pneumonia', 'emphysema'],
        'Hereditary': ['hereditary', 'familial', 'syndrome', 'congenital']
    }
    
    # Check each category
    for category, terms in categories.items():
        if any(term in disease_str for term in terms):
            return category
    
    return 'Other'

def extract_rs_ids(id_field):
    """Extract rs IDs from the ID field."""
    if pd.isna(id_field):
        return []
    # Check if this is an rs ID already
    if isinstance(id_field, str) and id_field.startswith('rs'):
        return [id_field]
    return []

def extract_clinvar_info(info_field):
    """Extract key information from ClinVar INFO field."""
    result = {}
    
    # Extract rs ID
    rs_match = re.search(r'RS=([^;]+)', info_field)
    if rs_match:
        result['rs_id'] = rs_match.group(1)
    
    # Extract clinical significance
    clnsig_match = re.search(r'CLNSIG=([^;]+)', info_field)
    if clnsig_match:
        result['clinical_significance'] = clnsig_match.group(1)
    
    # Extract disease names
    clndn_match = re.search(r'CLNDN=([^;]+)', info_field)
    if clndn_match:
        result['disease_names'] = clndn_match.group(1).replace('_', ' ')
    
    # Extract gene info
    gene_match = re.search(r'GENEINFO=([^;]+)', info_field)
    if gene_match:
        result['gene_info'] = gene_match.group(1)
    
    # Extract oncology significance if present
    onc_match = re.search(r'ONC=([^;]+)', info_field)
    if onc_match:
        result['oncology_significance'] = onc_match.group(1)
    
    return result

def extract_rs_ids_from_existing_variation(field):
    """Extract rs IDs from Existing_variation field that may contain multiple IDs."""
    if pd.isna(field):
        return []
    # Split by & or commas depending on format
    if '&' in field:
        variants = field.split('&')
    elif ',' in field:
        variants = field.split(',')
    else:
        variants = [field]
    # Extract only rs IDs
    rs_ids = [var for var in variants if var.startswith('rs')]
    return rs_ids

def load_variant_file(filepath, id):
    """Load variant file and extract wife's genotype."""
    print(f"Reading file: {filepath}")
    # Try different separators
    try:
        df = pd.read_csv(filepath, sep=';')
        print("Detected semicolon-separated file")
    except:
        try:
            df = pd.read_csv(filepath, sep='\t')
            print("Detected tab-separated file")
        except:
            try:
                df = pd.read_csv(filepath, sep=',')
                print("Detected comma-separated file")
            except Exception as e:
                print(f"Error reading file: {e}")
                raise
    
    print(f"Columns in file: {', '.join(df.columns[:10])}...")
    
    # Check if we have the expected ID column
    id_column = None
    rs_id_column = None
    
    # Look for appropriate columns to use for rs IDs
    if 'ID' in df.columns:
        id_column = 'ID'
    elif 'id' in df.columns:
        id_column = 'id'
    
    # Check for Existing_variation column which contains rs IDs
    if 'Existing_variation' in df.columns:
        rs_id_column = 'Existing_variation'
        print("Found Existing_variation column for rs IDs")
    else:
        # Check for rs IDs specifically in other columns
        for col in df.columns:
            if 'rs' in col.lower() or 'variation' in col.lower():
                rs_id_column = col
                print(f"Using {rs_id_column} for rs IDs")
                break
    
    # Choose the most appropriate column for rs IDs
    if rs_id_column:
        print(f"Using {rs_id_column} for rs IDs")
        df['rs_ids'] = df[rs_id_column].apply(extract_rs_ids)
    elif id_column:
        print(f"Using {id_column} for rs IDs")
        df['rs_ids'] = df[id_column].apply(extract_rs_ids)
    else:
        print("No suitable column found for rs IDs. Using basic ID column.")
        if 'ID' in df.columns:
            df['rs_ids'] = df['ID'].apply(extract_rs_ids)
        else:
            print("WARNING: No ID column found. Creating empty rs_ids column.")
            df['rs_ids'] = [[] for _ in range(len(df))]
    
    # Keep columns that might be useful
    cols_to_keep = []
    
    # Build list of columns to keep based on what's available
    useful_columns = ['ID', 'Chrom', 'Start', 'Ref', 'Alt', 
                      'CLIN_SIG', 'SYMBOL', 'Consequence', 'IMPACT', f'Fm_G4_{id}.GT']
    
    for col in useful_columns:
        if col in df.columns:
            cols_to_keep.append(col)
    
    # Always keep rs_ids
    cols_to_keep.append('rs_ids')
    
    # Keep only necessary columns that exist in the dataframe
    df = df[cols_to_keep]
    
    # Explode the dataframe so each rs ID gets its own row
    df = df.explode('rs_ids')
    
    # Drop rows with no rs IDs
    df = df.dropna(subset=['rs_ids'])
    
    print(f"Found {len(df)} rows with rs IDs")
    return df

def load_clinvar_pathogenic_variants(filepath):
    """Load pathogenic or likely pathogenic variants from ClinVar VCF."""
    print(f"Reading ClinVar file: {filepath}")
    
    # Read the VCF file, skipping header lines
    with open(filepath, 'r') as f:
        lines = [line for line in f if not line.startswith('##')]
    
    if not lines:
        print("No data found in ClinVar file")
        return pd.DataFrame()
    
    # Parse the header and data lines
    header = lines[0].strip().split('\t')
    print(f"ClinVar header: {header}")
    data_rows = []
    
    variant_count = 0
    pathogenic_count = 0
    rs_count = 0
    
    for line in tqdm(lines[1:], total=len(lines)-1):
        if line.strip():
            variant_count += 1
                
            fields = line.strip().split('\t')
            # Check if this entry contains clinical significance information
            if len(fields) >= 8:  # Ensure we have enough fields
                info_field = fields[7]
                
                # Extract RS ID
                rs_match = re.search(r'RS=([^;]+)', info_field)
                
                # Skip if there's no rs ID
                if not rs_match:
                    # Check if ID field has rs
                    if len(fields) >= 3 and fields[2].startswith('rs'):
                        # Use ID field as rs_id
                        rs_id = fields[2]
                    else:
                        continue
                else:
                    rs_id = "rs" + rs_match.group(1)
                    rs_count += 1
                
                # Extract clinical significance
                clnsig_match = re.search(r'CLNSIG=([^;]+)', info_field)
                
                if clnsig_match:
                    significance = clnsig_match.group(1).lower()
                    # Include pathogenic, likely pathogenic, and VUS variants
                    if ('pathogenic' in significance or 
                        'uncertain_significance' in significance):
                        pathogenic_count += 1
                        row_data = dict(zip(header, fields))
                        info_data = extract_clinvar_info(info_field)
                        
                        # Set rs_id explicitly
                        info_data['rs_id'] = rs_id
                        
                        row_data.update(info_data)
                        data_rows.append(row_data)
    
    print(f"Processed {variant_count} variants, found {pathogenic_count} pathogenic/VUS variants")
    
    # Create DataFrame from extracted data
    clinvar_df = pd.DataFrame(data_rows)
    
    print(f"Found {rs_count} variants with rs IDs")
    print(f"Found {pathogenic_count} pathogenic/VUS variants")
    
    if not clinvar_df.empty and 'rs_id' in clinvar_df.columns:
        # Ensure rs IDs are in standard format
        clinvar_df['rs_id'] = clinvar_df['rs_id'].apply(lambda x: x if x.startswith('rs') else f"rs{x}")
        print(f"Example rs IDs: {', '.join(clinvar_df['rs_id'].head().tolist())}")
    
    return clinvar_df

def analyze_disease_risk(variant_file, clinvar_file, output_file, id):
    """Analyze disease risk by matching variants with ClinVar data."""
    print(f"Loading variant file: {variant_file}")
    variant_df = load_variant_file(variant_file, id)
    
    print(f"Loading ClinVar file: {clinvar_file}")
    clinvar_df = load_clinvar_pathogenic_variants(clinvar_file)
    
    if clinvar_df.empty:
        print("No pathogenic variants found in ClinVar file")
        return
    
    # Make sure rs_id column exists in clinvar_df
    if 'rs_id' not in clinvar_df.columns:
        print("No rs IDs found in ClinVar file")
        return
    
    # Match variants by rs ID
    print("Matching variants...")
    matched_variants = pd.merge(
        variant_df, 
        clinvar_df,
        left_on='rs_ids',
        right_on='rs_id',
        how='inner'
    )
    
    if matched_variants.empty:
        print("No matches found between your variants and pathogenic ClinVar entries")
        return
    
    print(f"Found {len(matched_variants)} matched variants")
    
    # Focus on pathogenic/likely pathogenic variants
    pathogenic_terms = ['pathogenic', 'likely_pathogenic']
    pathogenic_pattern = '|'.join(pathogenic_terms)
    
    high_risk_variants = matched_variants[
        matched_variants['clinical_significance'].str.contains(pathogenic_pattern, 
                                                             case=False, 
                                                             na=False)
    ]
    
    # Focus on VUS (Variants of Uncertain Significance)
    vus_variants = matched_variants[
        matched_variants['clinical_significance'].str.contains('uncertain_significance', 
                                                             case=False, 
                                                             na=False)
    ]
    
    # Group by disease type for better organization
    if 'disease_names' in matched_variants.columns:
        # Create a helper column with disease categories
        matched_variants['disease_category'] = matched_variants['disease_names'].apply(
            lambda x: categorize_disease(x) if pd.notna(x) else 'Other'
        )
    
    # Save results
    print(f"Writing results to {output_file}")
    with pd.ExcelWriter(output_file) as writer:
        matched_variants.to_excel(writer, sheet_name='All Matches', index=False)
        
        if not high_risk_variants.empty:
            high_risk_variants.to_excel(writer, sheet_name='High Risk Variants', index=False)
            
            # Create sheets for common disease categories if we have sufficient data
            if len(high_risk_variants) > 2 and 'disease_category' in high_risk_variants.columns:
                disease_counts = high_risk_variants['disease_category'].value_counts()
                for disease_cat, count in disease_counts.items():
                    if count >= 2 and disease_cat != 'Other':  # Only create sheets for categories with multiple variants
                        category_variants = high_risk_variants[high_risk_variants['disease_category'] == disease_cat]
                        sheet_name = f"{disease_cat[:28]}"  # Truncate name to avoid Excel sheet name length limits
                        category_variants.to_excel(writer, sheet_name=sheet_name, index=False)
        
        if not vus_variants.empty:
            vus_variants.to_excel(writer, sheet_name='Uncertain Significance', index=False)
    
    print(f"Analysis complete. Found {len(matched_variants)} matched variants.")
    print(f"High risk variants: {len(high_risk_variants)}")
    print(f"Variants of uncertain significance: {len(vus_variants)}")
    
    print("\nTop disease categories found:")
    if 'disease_category' in matched_variants.columns and not matched_variants.empty:
        category_counts = matched_variants['disease_category'].value_counts()
        for cat, count in category_counts.items():
            if count > 0:
                print(f"  - {cat}: {count} variants")

In [8]:
id = '39'
variant_file = "../../data/wgs_all.csv"
clinvar_file = "../../data/clinvar.vcf"
output_file = f"disease_risk_analysis_{id}.xlsx"

analyze_disease_risk(variant_file, clinvar_file, output_file, id)

Loading variant file: ../../data/wgs_all.csv
Reading file: ../../data/wgs_all.csv
Detected semicolon-separated file
Columns in file: ID, Chrom, Start, Existing_variation, Ref, Alt, QUAL, ExcessHet, FS, MLEAC...
Found Existing_variation column for rs IDs
Using Existing_variation for rs IDs
Found 99247 rows with rs IDs
Loading ClinVar file: ../../data/clinvar.vcf
Reading ClinVar file: ../../data/clinvar.vcf
ClinVar header: ['#CHROM', 'POS', 'ID', 'REF', 'ALT', 'QUAL', 'FILTER', 'INFO']


100%|██████████| 3375801/3375801 [00:04<00:00, 681253.45it/s]


Processed 3375801 variants, found 815948 pathogenic/VUS variants
Found 1468104 variants with rs IDs
Found 815948 pathogenic/VUS variants
Example rs IDs: rs1640863258, rs1030245330, rs1329301928, rs2100289567, rs1381099827
Matching variants...
Found 1552 matched variants
Writing results to disease_risk_analysis_39.xlsx
Analysis complete. Found 1552 matched variants.
High risk variants: 1009
Variants of uncertain significance: 543

Top disease categories found:
  - Other: 819 variants
  - Hereditary: 439 variants
  - Neurological: 98 variants
  - Cardiovascular: 83 variants
  - Cancer: 72 variants
  - Metabolic: 21 variants
  - Respiratory: 12 variants
  - Autoimmune: 8 variants
